In [15]:
import os
from datetime import UTC, datetime, timedelta
from typing import Literal

from dotenv import load_dotenv
from google.genai import Client
from pydantic import BaseModel, Field

load_dotenv()

True

In [6]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1

In [2]:
class Message(BaseModel):
    role: str
    content: str
    pinned: bool


class Conversation(BaseModel):
    id: str
    messages: list[Message] = Field(default_factory=list)

In [ ]:
def build_history(conversation: Conversation, limit: int) -> list[dict]:
    pinned = []
    for message in conversation.messages:
        if message.pinned:
            pinned.append({"role": message.role, "content": message.content})

    history = []
    while limit > 0:  # limit replace with token budget or bussiness logic
        for message in conversation.messages:
            if not message.pinned:
                history.append({"role": message.role, "content": message.content})
                limit -= 1

    return pinned + history

### Task 2

In [7]:
class MemoryCandidate(BaseModel):
    content: str = Field(..., description="The content of the memory candidate.")
    category: Literal["preference", "fact", "event", "goal", "other"] = Field(
        ..., description="The category of the memory candidate."
    )
    confidence: float = Field(..., description="The confidence score of the memory candidate.")


class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


# Builds a prompt containing all supplied conversation messages.
def build_prompt(messages: list[Message]) -> str:
    prompt = (
        "You are a memory extraction assistant. "
        "You will be given a list of conversation messages. "
        "Your task is to extract potential memory candidates from the conversation. "
        "A memory candidate is a piece of information that could be useful for future interactions. "
        "For each memory candidate, provide the content, category (preference, fact, event, goal, other), and confidence score (0-1). "
        "Return the results in given format."
    )
    prompt += "\n\nHere are the conversation messages:\n"
    for msg in messages:
        prompt += f"{msg.role}: {msg.content}\n"
    return prompt


# Uses the configured LLM to extract typed memory candidates from messages.
async def extract_memory(conversations: list[Message]) -> list[MemoryCandidate]:
    prompt = build_prompt(conversations)
    response = await google_client.aio.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": list[MemoryCandidate]},
    )
    return response.parsed

### Task 3

completed in hands_on task 4.

### Task 4

In [8]:
def is_duplicate(new_memory, existing_memories, embedder, threshold=0.90):
    new_vec = embedder.embed(new_memory)

    for memory in existing_memories:
        old_vec = embedder.embed(memory)
        similarity = cosine_similarity(new_vec, old_vec)

        if similarity >= threshold:
            return True

    return False


def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b)) / (
        (sum(x * x for x in a) ** 0.5) * (sum(x * x for x in b) ** 0.5)
    )

### Task 5

In [9]:
class EpisodicMemoryStore:
    def __init__(self, embedder):
        self.embedder = embedder
        self.memories = []

    def add(self, action, result):
        self.memories.append(
            {"action": action, "result": result, "embedding": self.embedder.embed(action)}
        )

    def query(self, action, similar=False, threshold=0.85):
        # Exact action check
        for memory in self.memories:
            if memory["action"].strip().lower() == action.strip().lower():
                return [memory]

        if not similar:
            return []

        # Similar action search
        query_vec = self.embedder.embed(action)

        results = []
        for memory in self.memories:
            score = cosine_similarity(query_vec, memory["embedding"])

            if score >= threshold:
                results.append((score, memory))

        return sorted(results, reverse=True, key=lambda x: x[0])

### Task 6

In [ ]:
def prune_memories(memories, confidence_threshold=0.5, retention_days=90):
    cutoff = datetime.utcnow() - timedelta(days=retention_days)  # noqa : DTZ003

    active = []
    archived = []
    deleted = []

    for memory in memories:
        old = memory["last_used"] < cutoff
        low_confidence = memory["confidence"] < confidence_threshold

        if low_confidence:
            deleted.append(memory)

        elif old:
            archived.append(memory)

        else:
            active.append(memory)

    return {
        "active": active,
        "archived": archived,
        "deleted": deleted,
    }

### Task 7

In [ ]:
def test_deleted_memory_excluded_immediately(
    memory_store,
):  # use this memory store with real store to check
    memory = memory_store.add("User prefers Python")

    # Verify it is retrievable
    assert memory_store.retrieve("Python") == [memory]

    # Delete it
    memory_store.delete(memory.id)

    # Must NOT be returned immediately
    results = memory_store.retrieve("Python")

    assert memory not in results

### Task 8

In [14]:
# when constructing a prompt for the LLM, we want to include the long-term memory context in a structured format.
# This function takes a list of memory objects and formats them into a string that can be included in the prompt template directly as a string.


def format_memory_context(memories):
    if not memories:
        return ""

    lines = [
        "### LONG-TERM MEMORY",
        "The following information comes from previous interactions.",
        "It is memory, not retrieved document evidence.",
    ]

    for memory in memories:
        lines.append(f"- type: {memory['type']}, content: {memory['content']}")

    return "\n".join(lines)

### Task 9

In [17]:
def mark_accessed(memory):
    memory["access_count"] += 1
    memory["last_accessed_at"] = datetime.now(UTC)


def find_memories(memories, query):
    results = []
    for memory in memories:
        if query.lower() in memory["content"].lower():
            results.append(memory)
    return results


def retrieve(memories, query):
    results = find_memories(memories, query)

    # Update only successfully retrieved memories
    for memory in results:
        mark_accessed(memory)

    return results

### Task 10

implemented in exercise_test.py